# Week 2: Your first model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GDG-TUM/AI-ML-Track-GDG-on-Campus-TUM/blob/main/weekly-sessions/week-02-your-first-model/first_model.ipynb)

**GDG on Campus TUM · AI/ML Track**

Today a computer learns to read handwriting. You will:

1. Look at the data (always step one)
2. Set a **baseline** to beat
3. Train real models with scikit-learn
4. Study where they get it wrong

No downloads needed. The dataset ships inside scikit-learn.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 0  # so that everyone in the room gets the same answer

## 1. Look at the data

`load_digits` has 1,797 tiny 8×8 pixel pictures of handwritten digits 0 to 9. Each picture is flattened into **64 numbers** (one per pixel, 0 = white to 16 = black). The **label** is the digit it shows.

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
print("X (features):", X.shape, "-> 1797 pictures, 64 pixels each")
print("y (labels)  :", y.shape, "-> the true digit for each picture")

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, image, label in zip(axes.ravel(), digits.images, digits.target, strict=False):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(f"label {label}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Are the classes balanced? (about the same number of each digit?)
pd.Series(y).value_counts().sort_index().to_frame("count").T

## 2. Split: keep a secret exam

If you study from the exam paper and then sit the same exam, a good mark proves nothing. Models are the same. We hide 25% of the data (the **test set**) and only look at it at the very end.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)
print("train:", X_train.shape[0], "pictures | test:", X_test.shape[0], "pictures")

## 3. Baseline: the dumbest possible model

Before you build anything clever, ask: *what score could I get by doing nothing smart?* A model that always guesses the most common digit gives us the number to beat. If your model cannot clearly beat the baseline, it has learned nothing.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_acc = accuracy_score(y_test, baseline.predict(X_test))
print(f"baseline accuracy: {baseline_acc:.3f}  (roughly 1 in 10, as expected)")

## 4. Real models

scikit-learn gives every model the same three verbs: **`fit`** (learn from training data), **`predict`** (guess on new data), **`score`** (how good was that). Learn them once, use them everywhere.

We compare three very different ideas:

| Model | Idea in one line |
|-------|------------------|
| **Logistic regression** | Draw straight boundaries between classes |
| **k-nearest neighbours** | "You look like these 5 pictures, so you are probably the same digit" |
| **Decision tree** | A flowchart of yes/no questions about pixels |

In [ ]:
models = {
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "k-nearest neighbours": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "decision tree": DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    rows.append(
        {
            "model": name,
            "train accuracy": model.score(X_train, y_train),
            "test accuracy": model.score(X_test, y_test),
        }
    )

results = pd.DataFrame(rows).set_index("model").round(3)
results.loc["baseline (most frequent)"] = [
    round(baseline.score(X_train, y_train), 3),
    round(baseline_acc, 3),
]
results.sort_values("test accuracy", ascending=False)

Look at the **train** versus **test** columns. When train accuracy is much higher than test accuracy the model has partly *memorised* the training pictures instead of learning what a "3" looks like. That is called **overfitting**, and Week 3 is all about catching it.

## 5. Where does it go wrong?

Accuracy is one number. A **confusion matrix** shows *which* digits get mixed up: rows are the truth, columns are the guess. A perfect model has numbers only on the diagonal.

In [ ]:
best = models["k-nearest neighbours"]
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(best, X_test, y_test, cmap="Blues", colorbar=False, ax=ax)
ax.set_title("k-nearest neighbours: truth vs guess")
plt.show()

In [ ]:
print(classification_report(y_test, best.predict(X_test), digits=3))

Now look at the actual mistakes. Would *you* have got them right?

In [ ]:
pred = best.predict(X_test)
wrong = np.where(pred != y_test)[0]
print(f"{len(wrong)} mistakes out of {len(y_test)} test pictures")

fig, axes = plt.subplots(1, min(8, len(wrong)), figsize=(12, 2.2))
for ax, idx in zip(np.atleast_1d(axes), wrong[:8], strict=False):
    ax.imshow(X_test[idx].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"true {y_test[idx]}\nguess {pred[idx]}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Self-check: a real model must clearly beat the baseline.
assert results.loc["k-nearest neighbours", "test accuracy"] > baseline_acc + 0.5
print("✅ Your model beats the baseline by a mile. Nice.")

## 🏁 Take-home challenge

1. Change `n_neighbors` (try 1, 3, 15, 50). What happens to train and test accuracy?
2. Add a fourth model, for example `RandomForestClassifier` from `sklearn.ensemble`, and add it to the table.
3. Remove `StandardScaler` from the logistic regression pipeline. Does it matter here? Why might it matter in other datasets?
4. Write **five sentences**: which model would you use and why? What would worry you before using it on real handwriting from our campus forms?

Post your table and your five sentences in the track channel.

**Next week:** how do you know a score is honest? Cross-validation, leakage and the trap of accuracy.